# Chapitre 4 — EDA diagnostique et qualité des données

**Durée estimée : 8-10 heures**

---

## Objectifs d'apprentissage

À la fin de ce chapitre, vous serez capable de :

1. **Distinguer** l'EDA diagnostique (trouver les problèmes) de l'EDA analytique (comprendre les patterns)
2. **Évaluer** la qualité des données selon les 6 dimensions standardisées
3. **Détecter** les valeurs manquantes, doublons et outliers avec les outils pandas appropriés
4. **Documenter** les problèmes identifiés dans une checklist de qualité priorisée

---

## 🎯 Le Hook : Le satellite qui s'est crashé à cause d'une virgule

En 1999, la NASA a perdu la sonde Mars Climate Orbiter — un investissement de **125 millions de dollars** — à cause d'une erreur de données. Un système utilisait des mesures en **livres-force** tandis qu'un autre attendait des **newtons**. Personne n'a vérifié la cohérence des unités.

La sonde s'est désintégrée dans l'atmosphère martienne.

Ce n'était pas un bug de code. C'était un **problème de qualité des données**.

Avant de nettoyer, transformer ou analyser vos données, vous devez d'abord **diagnostiquer** ce qui ne va pas. C'est l'objet de ce chapitre.

> 💭 **Question Socratique #1** : À votre avis, pourquoi les équipes de la NASA — composées d'ingénieurs brillants — n'ont-elles pas détecté cette incohérence d'unités ? Quel processus aurait pu prévenir ce désastre ?

---

# 📖 PARTIE THÉORIQUE

---

## 4.1 Objectif de l'EDA diagnostique

### Diagnostique vs Analytique : deux missions différentes

| Aspect | EDA Diagnostique (Chapitre 4) | EDA Analytique (Chapitre 7) |
|--------|------------------------------|----------------------------|
| **Question** | *"Qu'est-ce qui ne va pas ?"* | *"Que disent les données ?"* |
| **Objectif** | Trouver les problèmes de qualité | Comprendre les patterns |
| **Timing** | Avant le nettoyage | Après le nettoyage |
| **Focus** | Valeurs manquantes, erreurs, incohérences | Corrélations, tendances, segments |
| **Mindset** | Détective / Auditeur | Explorateur / Scientifique |

```
Données brutes → [EDA DIAGNOSTIQUE] → [NETTOYAGE] → [EDA ANALYTIQUE] → Insights
                  (Vous êtes ici)
```

### Pourquoi ne pas sauter cette étape ?

**Erreur courante :** Se lancer directement dans l'analyse sans vérifier la qualité.

**Conséquences :**
- Conclusions fausses basées sur des données erronées
- Temps perdu à recommencer
- Décisions business incorrectes
- Modèles ML qui ne fonctionnent pas en production

**Règle d'or :** Passez 20% de votre temps à diagnostiquer pour économiser 80% de corrections futures.

### ✍️ Exercice 4.1 : Diagnostic ou Analytique ? (5 min)

Classez chaque action dans la bonne catégorie :

| Action | Diagnostique | Analytique |
|--------|--------------|------------|
| Calculer la corrélation entre prix et ventes | ○ | ○ |
| Compter les valeurs manquantes par colonne | ○ | ○ |
| Identifier les segments de clients | ○ | ○ |
| Vérifier si les dates sont au bon format | ○ | ○ |
| Analyser la distribution des âges | ○ | ○ |
| Chercher les doublons dans les emails | ○ | ○ |

---

## 4.2 Les 6 dimensions de la qualité des données

### Framework standardisé

La qualité des données s'évalue selon **6 dimensions principales** reconnues par l'industrie :

```
┌─────────────────────────────────────────────────────────────────────┐
│                    DIMENSIONS DE QUALITÉ                            │
├─────────────────┬─────────────────┬─────────────────────────────────┤
│  COMPLÉTUDE     │   UNICITÉ       │         COHÉRENCE               │
│  (Completeness) │   (Uniqueness)  │       (Consistency)             │
│                 │                 │                                 │
│  Toutes les     │  Pas de         │  Mêmes valeurs dans             │
│  données sont   │  doublons       │  différents systèmes            │
│  présentes ?    │                 │                                 │
├─────────────────┼─────────────────┼─────────────────────────────────┤
│  EXACTITUDE     │   VALIDITÉ      │         FRAÎCHEUR               │
│  (Accuracy)     │   (Validity)    │       (Timeliness)              │
│                 │                 │                                 │
│  Les données    │  Format et      │  Données à jour,                │
│  reflètent la   │  règles         │  pas obsolètes                  │
│  réalité ?      │  respectés ?    │                                 │
└─────────────────┴─────────────────┴─────────────────────────────────┘
```

*(Source : [IBM - Data Quality Dimensions](https://www.ibm.com/think/topics/data-quality-dimensions))*

### 1. Complétude (Completeness)

**Question :** Toutes les données requises sont-elles présentes ?

**Exemples de problèmes :**
- Client sans adresse email
- Commande sans date de livraison
- Produit sans prix

**Métrique :** `% de valeurs non-nulles`

### 2. Unicité (Uniqueness)

**Question :** Y a-t-il des enregistrements en double ?

**Exemples de problèmes :**
- Même client enregistré 3 fois
- Même transaction comptée deux fois
- Doublons dus à des imports multiples

**Métrique :** `% d'enregistrements uniques`

### 3. Cohérence (Consistency)

**Question :** Les données sont-elles logiquement cohérentes ?

**Exemples de problèmes :**
- Date de naissance > date d'inscription
- Ville "Paris" avec code postal "69000" (Lyon)
- Total ≠ somme des lignes

**Métrique :** `% de règles de cohérence respectées`

### 4. Exactitude (Accuracy)

**Question :** Les données reflètent-elles la réalité ?

**Exemples de problèmes :**
- Âge de 150 ans (erreur de saisie)
- Température de -100°C à Paris
- Salaire négatif

**Métrique :** `% de valeurs dans les plages attendues`

### 5. Validité (Validity)

**Question :** Les données respectent-elles le format attendu ?

**Exemples de problèmes :**
- Email sans "@"
- Code postal avec des lettres
- Date au format "31/13/2024"

**Métrique :** `% de valeurs conformes au format`

### 6. Fraîcheur (Timeliness)

**Question :** Les données sont-elles à jour ?

**Exemples de problèmes :**
- Adresse d'un client déménagé il y a 2 ans
- Prix catalogue de 2019
- Statut de commande non mis à jour

**Métrique :** `Âge moyen des données`

### ✍️ Exercice 4.2 : Identifier les dimensions (10 min)

Pour chaque problème, identifiez la dimension de qualité concernée :

| Problème | Dimension |
|----------|----------|
| 15% des clients n'ont pas de numéro de téléphone | _____ |
| Le même produit apparaît 3 fois avec des prix différents | _____ |
| Un client a une date de naissance en 2050 | _____ |
| Le code postal contient des caractères spéciaux | _____ |
| L'adresse d'un entrepôt fermé depuis 2 ans est encore active | _____ |
| Le total de la facture ne correspond pas à la somme des lignes | _____ |

> 💭 **Question Socratique #2** : Une entreprise peut-elle avoir des données 100% complètes mais de mauvaise qualité ? Donnez un exemple concret.

---

# 🖥️ PARTIE PRATIQUE

---

## 4.3 Statistiques descriptives de diagnostic

### Les commandes essentielles

In [ ]:
import pandas as pd
import numpy as np

# Création d'un DataFrame de démonstration
np.random.seed(42)
df = pd.DataFrame({
    'client_id': range(1, 101),
    'nom': [f'Client_{i}' for i in range(1, 101)],
    'email': [f'client{i}@test.com' if i % 10 != 0 else None for i in range(1, 101)],
    'age': np.random.randint(18, 70, 100),
    'montant_achats': np.random.uniform(100, 10000, 100).round(2)
})

# Introduisons quelques problèmes
df.loc[5, 'age'] = 150  # Âge invalide
df.loc[10, 'age'] = -5  # Âge négatif
df.loc[95:99, 'montant_achats'] = np.nan  # Valeurs manquantes

print("DataFrame de démonstration créé avec des problèmes de qualité.")

In [ ]:
# 1. Dimensions
print(f"📊 Shape : {df.shape[0]} lignes × {df.shape[1]} colonnes")

In [ ]:
# 2. Types de données
print("📝 Types :")
print(df.dtypes)

In [ ]:
# 3. Aperçu
print("👀 Premières lignes :")
df.head()

In [ ]:
# 4. Résumé complet
print("📋 Info :")
df.info()

In [ ]:
# 5. Statistiques numériques
print("📈 Describe :")
df.describe()

In [ ]:
# 6. Statistiques catégorielles
print("🏷️ Valeurs uniques par colonne :")
for col in df.columns:
    print(f"{col}: {df[col].nunique()} valeurs uniques")

### Tableau de référence

| Commande | Information | Utilité diagnostique |
|----------|-------------|---------------------|
| `df.shape` | (lignes, colonnes) | Volume attendu ? |
| `df.dtypes` | Types par colonne | Types corrects ? |
| `df.head()` | 5 premières lignes | Aperçu visuel |
| `df.info()` | Résumé + mémoire | Valeurs non-null |
| `df.describe()` | Stats numériques | Min/max aberrants ? |
| `df.nunique()` | Valeurs uniques | Cardinalité |
| `df.value_counts()` | Distribution | Catégories inattendues ? |

### Mesurer la complétude et l'unicité

In [ ]:
# Calculer la complétude par colonne
completude = (1 - df.isnull().mean()) * 100
print("Complétude par colonne (%) :")
print(completude.sort_values())

In [ ]:
# Taux d'unicité
unicite = (1 - df.duplicated().mean()) * 100
print(f"Unicité globale : {unicite:.2f}%")

### ✍️ Exercice 4.3 : Exploration initiale (15 min)

Analysez ce DataFrame et identifiez les problèmes potentiels :

In [ ]:
import pandas as pd
import numpy as np

# Données simulées avec des problèmes
df_exercice = pd.DataFrame({
    'id': [1, 2, 3, 4, 5, 5],
    'nom': ['Alice', 'Bob', 'Charlie', 'David', 'Eve', 'Eve'],
    'age': [25, 150, 35, -5, 28, 28],
    'email': ['alice@test.com', 'bob', 'charlie@test.com', None, 'eve@test.com', 'eve@test.com'],
    'salaire': [50000, 55000, np.nan, 60000, 45000, 45000],
    'date_inscription': ['2024-01-15', '2024-02-30', '2024-03-10', '2024-04-05', '2024-05-12', '2024-05-12']
})

# Votre exploration
print("1. Shape :", df_exercice.shape)
print("\n2. Types :\n", df_exercice.dtypes)
print("\n3. Valeurs manquantes :\n", df_exercice.isnull().sum())
print("\n4. Doublons :", df_exercice.duplicated().sum())
print("\n5. Stats numériques :\n", df_exercice.describe())

In [ ]:
# Questions :
# a) Combien de problèmes de COMPLÉTUDE identifiez-vous ?
print("a) Problèmes de COMPLÉTUDE :")
print(f"   - email manquant : {df_exercice['email'].isnull().sum()} valeur(s)")
print(f"   - salaire manquant : {df_exercice['salaire'].isnull().sum()} valeur(s)")

# b) Combien de problèmes d'EXACTITUDE ?
print("\nb) Problèmes d'EXACTITUDE :")
print(f"   - âge > 120 : {(df_exercice['age'] > 120).sum()} valeur(s)")
print(f"   - âge < 0 : {(df_exercice['age'] < 0).sum()} valeur(s)")

# c) Combien de problèmes de VALIDITÉ ?
print("\nc) Problèmes de VALIDITÉ :")
emails_sans_at = df_exercice[~df_exercice['email'].str.contains('@', na=False)]
print(f"   - emails sans @ : {len(emails_sans_at)} valeur(s)")

# d) Combien de problèmes d'UNICITÉ ?
print("\nd) Problèmes d'UNICITÉ :")
print(f"   - doublons exacts : {df_exercice.duplicated().sum()} ligne(s)")
print(f"   - doublons sur id : {df_exercice.duplicated(subset=['id']).sum()} ligne(s)")

---

## 4.4 Détection des valeurs manquantes

### Comprendre les patterns de missing

Les valeurs manquantes ne sont pas toutes égales. Il existe trois types :

| Type | Définition | Exemple |
|------|------------|--------|
| **MCAR** (Missing Completely At Random) | Manque aléatoire, aucun pattern | Erreur de saisie ponctuelle |
| **MAR** (Missing At Random) | Manque lié à d'autres variables | Revenus manquants surtout chez les jeunes |
| **MNAR** (Missing Not At Random) | Manque lié à la valeur elle-même | Revenus élevés non déclarés volontairement |

**Pourquoi c'est important ?** Le type de missing influence la stratégie de traitement (Chapitre 5).

### Quantifier les valeurs manquantes

In [ ]:
# Créer un DataFrame avec des missing
np.random.seed(42)
df_missing = pd.DataFrame({
    'A': [1, 2, np.nan, 4, 5, np.nan, 7, 8, 9, 10],
    'B': [np.nan, 2, 3, np.nan, 5, 6, np.nan, 8, np.nan, 10],
    'C': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
})

print("DataFrame avec valeurs manquantes :")
df_missing

In [ ]:
# Nombre de valeurs manquantes par colonne
print("Valeurs manquantes par colonne :")
print(df_missing.isnull().sum())

In [ ]:
# Pourcentage de valeurs manquantes
print("\nPourcentage de valeurs manquantes :")
print((df_missing.isnull().mean() * 100).round(2))

In [ ]:
# Lignes avec au moins une valeur manquante
lignes_incompletes = df_missing[df_missing.isnull().any(axis=1)]
print(f"\nLignes incomplètes : {len(lignes_incompletes)} ({len(lignes_incompletes)/len(df_missing)*100:.1f}%)")
print(lignes_incompletes)

### Visualiser avec missingno

In [ ]:
# pip install missingno
import missingno as msno
import matplotlib.pyplot as plt

# Créer un DataFrame plus grand pour mieux visualiser
np.random.seed(42)
n = 200
df_viz = pd.DataFrame({
    'col_A': np.where(np.random.random(n) < 0.1, np.nan, np.random.randn(n)),
    'col_B': np.where(np.random.random(n) < 0.25, np.nan, np.random.randn(n)),
    'col_C': np.where(np.random.random(n) < 0.05, np.nan, np.random.randn(n)),
    'col_D': np.random.randn(n),  # Pas de missing
    'col_E': np.where(np.random.random(n) < 0.3, np.nan, np.random.randn(n))
})

print(f"DataFrame créé : {df_viz.shape}")
print("\nValeurs manquantes :")
print(df_viz.isnull().sum())

In [ ]:
# Matrice de valeurs manquantes
msno.matrix(df_viz)
plt.title("Pattern des valeurs manquantes")
plt.show()

In [ ]:
# Heatmap de corrélation entre missing
msno.heatmap(df_viz)
plt.title("Corrélation entre valeurs manquantes")
plt.show()

**Interprétation de la matrice :**
- Blanc = valeur manquante
- Noir = valeur présente
- Patterns verticaux = colonnes problématiques
- Patterns horizontaux = lignes problématiques

### ✍️ Exercice 4.4 : Analyse des missing (15 min)

In [ ]:
import pandas as pd
import numpy as np

# Données avec patterns de missing
np.random.seed(42)
n = 1000

df_pattern = pd.DataFrame({
    'age': np.random.randint(18, 80, n),
    'revenu': np.random.randint(20000, 100000, n).astype(float),
    'education': np.random.choice(['Bac', 'Licence', 'Master', 'Doctorat'], n)
})

# Introduire des missing avec pattern
# Les jeunes (< 25 ans) ont souvent des revenus manquants
mask_jeunes = df_pattern['age'] < 25
random_mask = np.random.random(n) < 0.4
df_pattern.loc[mask_jeunes & random_mask, 'revenu'] = np.nan

# L'éducation est parfois manquante aléatoirement
random_mask_edu = np.random.random(n) < 0.05
df_pattern.loc[random_mask_edu, 'education'] = np.nan

print("DataFrame créé avec patterns de missing.")

In [ ]:
# Analyse
print("Valeurs manquantes par colonne :")
print(df_pattern.isnull().sum())

print("\nPourcentage de revenus manquants par tranche d'âge :")
df_pattern['tranche_age'] = pd.cut(df_pattern['age'], bins=[18, 25, 35, 50, 80], labels=['18-25', '26-35', '36-50', '51+'])
print(df_pattern.groupby('tranche_age')['revenu'].apply(lambda x: x.isnull().mean() * 100).round(2))

In [ ]:
# Questions :
# 1. Le missing sur 'revenu' est-il MCAR, MAR ou MNAR ?
print("1. Type de missing pour 'revenu' :")
print("   → MAR (Missing At Random) - les jeunes ont plus de valeurs manquantes")

# 2. Le missing sur 'education' est-il MCAR, MAR ou MNAR ?
print("\n2. Type de missing pour 'education' :")
print("   → MCAR (Missing Completely At Random) - 5% aléatoire")

# 3. Quelle stratégie de traitement suggéreriez-vous ?
print("\n3. Stratégies suggérées :")
print("   - revenu : imputation par groupe d'âge (médiane par tranche)")
print("   - education : imputation par mode ou suppression des lignes")

---

## 4.5 Détection des doublons

### Types de doublons

| Type | Définition | Exemple |
|------|------------|--------|
| **Exact** | Lignes 100% identiques | Import dupliqué |
| **Quasi-doublon** | Même entité, petites différences | "Jean Dupont" vs "Jean DUPONT" |
| **Doublon partiel** | Même clé, valeurs différentes | Même client avec 2 adresses |

### Détecter les doublons exacts

In [ ]:
# Créer un DataFrame avec des doublons
df_doublons = pd.DataFrame({
    'id': [1, 2, 3, 4, 5, 1, 2],
    'nom': ['Alice', 'Bob', 'Charlie', 'David', 'Eve', 'Alice', 'Bob'],
    'email': ['a@t.com', 'b@t.com', 'c@t.com', 'd@t.com', 'e@t.com', 'a@t.com', 'b@t.com'],
    'age': [25, 30, 35, 40, 28, 25, 30]
})

print("DataFrame avec doublons :")
df_doublons

In [ ]:
# Nombre de doublons
print(f"Doublons exacts : {df_doublons.duplicated().sum()}")

In [ ]:
# Voir les doublons (keep=False montre tous les doublons)
doublons = df_doublons[df_doublons.duplicated(keep=False)]
print("Lignes dupliquées :")
doublons.sort_values(by=['id'])

In [ ]:
# Doublons sur une clé spécifique
doublons_id = df_doublons[df_doublons.duplicated(subset=['id'], keep=False)]
print(f"\nDoublons sur 'id' : {df_doublons.duplicated(subset=['id']).sum()}")
print(doublons_id)

In [ ]:
# Doublons sur email
doublons_email = df_doublons[df_doublons.duplicated(subset=['email'], keep=False)]
print(f"Emails dupliqués : {doublons_email['email'].nunique()}")

### Identifier les clés candidates

In [ ]:
def verifier_cle_unique(df, colonnes):
    """Vérifie si un ensemble de colonnes peut servir de clé unique."""
    doublons = df.duplicated(subset=colonnes)
    if doublons.sum() == 0:
        print(f"✅ {colonnes} peut servir de clé unique")
    else:
        print(f"❌ {colonnes} a {doublons.sum()} doublons")

# Tests
verifier_cle_unique(df_doublons, ['id'])
verifier_cle_unique(df_doublons, ['email'])
verifier_cle_unique(df_doublons, ['nom', 'age'])

### ✍️ Exercice 4.5 : Chasse aux doublons (10 min)

In [ ]:
import pandas as pd

df_chasse = pd.DataFrame({
    'client_id': [1, 2, 3, 4, 5, 1],
    'nom': ['Alice Martin', 'Bob Dupont', 'Charlie Brown', 'David Lee', 'Eve Wilson', 'Alice MARTIN'],
    'email': ['alice@test.com', 'bob@test.com', 'charlie@test.com', 'david@test.com', 'eve@test.com', 'alice@test.com'],
    'date_achat': ['2024-01-15', '2024-01-16', '2024-01-17', '2024-01-18', '2024-01-19', '2024-01-20'],
    'montant': [100, 200, 150, 300, 250, 100]
})

print("DataFrame :")
df_chasse

In [ ]:
# Votre analyse
# 1. Combien de doublons exacts ?
print("Doublons exacts :", df_chasse.duplicated().sum())

# 2. Combien de doublons sur client_id ?
print("Doublons client_id :", df_chasse.duplicated(subset=['client_id']).sum())

# 3. Combien de doublons sur email ?
print("Doublons email :", df_chasse.duplicated(subset=['email']).sum())

# 4. Y a-t-il un quasi-doublon sur le nom ? Comment le détecteriez-vous ?
print("\nDétection quasi-doublons sur nom (normalisation) :")
df_chasse['nom_normalise'] = df_chasse['nom'].str.lower().str.strip()
print("Doublons après normalisation :", df_chasse.duplicated(subset=['nom_normalise']).sum())

> 💭 **Question Socratique #3** : Si deux lignes ont le même email mais des noms légèrement différents ("Jean Dupont" vs "jean dupont"), est-ce un doublon à supprimer ou deux entrées légitimes ? Comment décideriez-vous ?

---

## 4.6 Détection des outliers

### Qu'est-ce qu'un outlier ?

Un **outlier** (valeur aberrante) est une observation qui s'écarte significativement des autres.

**Attention :** Un outlier n'est pas toujours une erreur !
- ❌ Erreur : Âge de 200 ans (impossible)
- ✅ Valeur extrême légitime : Salaire de 500k€ (rare mais réel)

### Méthode 1 : IQR (Interquartile Range)

In [ ]:
def detecter_outliers_iqr(series, multiplicateur=1.5):
    """Détecte les outliers avec la méthode IQR."""
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    
    borne_inf = Q1 - multiplicateur * IQR
    borne_sup = Q3 + multiplicateur * IQR
    
    outliers = series[(series < borne_inf) | (series > borne_sup)]
    return outliers, borne_inf, borne_sup

# Créer des données avec outliers
np.random.seed(42)
salaires = np.concatenate([
    np.random.normal(50000, 10000, 100),  # Salaires normaux
    np.array([150000, 200000, 5000])  # Outliers
])
df_outliers = pd.DataFrame({'salaire': salaires})

# Application
outliers, b_inf, b_sup = detecter_outliers_iqr(df_outliers['salaire'])
print(f"Bornes : [{b_inf:.0f}, {b_sup:.0f}]")
print(f"Outliers : {len(outliers)} ({len(outliers)/len(df_outliers)*100:.1f}%)")
print(f"Valeurs : {outliers.values}")

### Méthode 2 : Z-Score

In [ ]:
from scipy import stats

def detecter_outliers_zscore(series, seuil=3):
    """Détecte les outliers avec le Z-score."""
    z_scores = stats.zscore(series.dropna())
    outliers_mask = abs(z_scores) > seuil
    return series.dropna()[outliers_mask]

# Application
outliers_z = detecter_outliers_zscore(df_outliers['salaire'])
print(f"Outliers (|Z| > 3) : {len(outliers_z)}")
print(f"Valeurs : {outliers_z.values}")

**Interprétation du Z-score :**
- |Z| > 2 : Valeur inhabituelle (~5% des données)
- |Z| > 3 : Valeur très rare (~0.3%)
- |Z| > 4 : Valeur extrêmement rare

### Comparaison des méthodes

| Méthode | Avantages | Inconvénients | Quand l'utiliser |
|---------|-----------|---------------|------------------|
| **IQR** | Robuste, pas d'hypothèse | Conservateur | Données asymétriques |
| **Z-Score** | Simple, interprétable | Sensible aux extrêmes | Données normales |

### ✍️ Exercice 4.6 : Détection d'outliers (15 min)

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats

# Données de salaires avec outliers
np.random.seed(42)
salaires = np.concatenate([
    np.random.normal(50000, 10000, 950),  # Salaires normaux
    np.array([150000, 200000, -5000, 300000, 250000])  # Outliers
])
df_ex = pd.DataFrame({'salaire': salaires})

print(f"Statistiques descriptives :")
print(df_ex['salaire'].describe())

In [ ]:
# 1. Méthode IQR
Q1 = df_ex['salaire'].quantile(0.25)
Q3 = df_ex['salaire'].quantile(0.75)
IQR = Q3 - Q1
borne_inf = Q1 - 1.5 * IQR
borne_sup = Q3 + 1.5 * IQR

outliers_iqr = df_ex[(df_ex['salaire'] < borne_inf) | (df_ex['salaire'] > borne_sup)]
print(f"Méthode IQR :")
print(f"  Bornes : [{borne_inf:.0f}, {borne_sup:.0f}]")
print(f"  Outliers : {len(outliers_iqr)}")

In [ ]:
# 2. Méthode Z-Score
z_scores = stats.zscore(df_ex['salaire'])
outliers_zscore = df_ex[abs(z_scores) > 3]
print(f"\nMéthode Z-Score :")
print(f"  Outliers (|Z| > 3) : {len(outliers_zscore)}")

In [ ]:
# Questions :
print("\n--- Réponses aux questions ---")

# a) Pourquoi les deux méthodes donnent-elles des résultats différents ?
print("\na) Différence entre méthodes :")
print("   - IQR est basé sur les quartiles (robuste aux outliers)")
print("   - Z-Score est basé sur moyenne/écart-type (sensible aux outliers)")
print("   - Les outliers extrêmes gonflent l'écart-type, rendant Z-Score moins sensible")

# b) Le salaire de -5000 est-il une erreur ou une valeur légitime ?
print("\nb) Salaire de -5000 :")
print("   → C'est une ERREUR car un salaire ne peut pas être négatif")
print("   → Ce type d'outlier est une erreur de saisie ou de conversion")

# c) Quelle méthode préférez-vous ici et pourquoi ?
print("\nc) Méthode préférée :")
print("   → IQR est préférable car :")
print("   - Plus robuste face aux valeurs extrêmes")
print("   - Détecte mieux les outliers quand la distribution est perturbée")

---

## 4.7 Visualisations de diagnostic

### Histogrammes pour les distributions

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Créer des données pour la visualisation
np.random.seed(42)
df_viz = pd.DataFrame({
    'age': np.concatenate([np.random.randint(18, 70, 97), [150, -5, 200]]),
    'salaire': np.concatenate([np.random.normal(50000, 15000, 95), [np.nan]*5]),
    'categorie': np.random.choice(['A', 'B', 'C'], 100)
})

print("Données préparées pour la visualisation")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Distribution des âges
df_viz['age'].hist(bins=30, ax=axes[0], edgecolor='black')
axes[0].set_title('Distribution des âges')
axes[0].set_xlabel('Âge')

# Distribution des salaires
df_viz['salaire'].dropna().hist(bins=30, ax=axes[1], edgecolor='black')
axes[1].set_title('Distribution des salaires')
axes[1].set_xlabel('Salaire')

# Distribution catégorielle
df_viz['categorie'].value_counts().plot(kind='bar', ax=axes[2], edgecolor='black')
axes[2].set_title('Distribution des catégories')

plt.tight_layout()
plt.show()

### Boxplots pour les outliers

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Boxplot simple
sns.boxplot(y=df_viz['age'], ax=axes[0])
axes[0].set_title('Boxplot des âges')

# Boxplot des salaires
sns.boxplot(y=df_viz['salaire'].dropna(), ax=axes[1])
axes[1].set_title('Boxplot des salaires')

plt.tight_layout()
plt.show()

**Lecture d'un boxplot :**
- Boîte = Q1 à Q3 (50% des données)
- Ligne centrale = Médiane
- Moustaches = 1.5 × IQR
- Points = Outliers

### Heatmap pour les valeurs manquantes

In [ ]:
# Créer un DataFrame avec plus de missing pour mieux visualiser
np.random.seed(42)
df_heatmap = pd.DataFrame({
    'col_A': np.where(np.random.random(50) < 0.15, np.nan, np.random.randn(50)),
    'col_B': np.where(np.random.random(50) < 0.30, np.nan, np.random.randn(50)),
    'col_C': np.where(np.random.random(50) < 0.05, np.nan, np.random.randn(50)),
    'col_D': np.random.randn(50),
    'col_E': np.where(np.random.random(50) < 0.25, np.nan, np.random.randn(50))
})

# Créer une matrice de missing (True/False)
missing_matrix = df_heatmap.isnull()

# Heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(missing_matrix, cbar=True, yticklabels=False, cmap='YlOrRd')
plt.title('Carte des valeurs manquantes (jaune = présent, rouge = manquant)')
plt.show()

### ✍️ Exercice 4.7 : Dashboard de diagnostic (20 min)

Créez un dashboard de diagnostic complet pour un DataFrame :

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Données simulées
np.random.seed(42)
n = 1000
df_dashboard = pd.DataFrame({
    'age': np.concatenate([np.random.randint(18, 70, 997), [150, -5, 200]]),
    'revenu': np.concatenate([np.random.normal(50000, 15000, 900), [np.nan]*100]),
    'categorie': np.random.choice(['A', 'B', 'C', None], 1000, p=[0.3, 0.3, 0.3, 0.1])
})

print("Données créées pour le dashboard")
print(f"Shape : {df_dashboard.shape}")

In [ ]:
# Dashboard
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Distribution des âges avec outliers
axes[0, 0].hist(df_dashboard['age'].dropna(), bins=30, edgecolor='black', alpha=0.7)
axes[0, 0].set_title('Distribution des âges')
axes[0, 0].axvline(df_dashboard['age'].median(), color='red', linestyle='--', label=f'Médiane: {df_dashboard["age"].median():.0f}')
axes[0, 0].legend()

# 2. Boxplot des revenus
sns.boxplot(y=df_dashboard['revenu'].dropna(), ax=axes[0, 1])
axes[0, 1].set_title('Boxplot des revenus')

# 3. Valeurs manquantes par colonne
missing_pct = (df_dashboard.isnull().mean() * 100)
missing_pct.plot(kind='bar', ax=axes[1, 0], color='coral', edgecolor='black')
axes[1, 0].set_title('% Valeurs manquantes par colonne')
axes[1, 0].set_ylabel('Pourcentage')
axes[1, 0].set_xticklabels(axes[1, 0].get_xticklabels(), rotation=0)

# 4. Distribution des catégories
cat_counts = df_dashboard['categorie'].value_counts(dropna=False)
cat_counts.plot(kind='pie', ax=axes[1, 1], autopct='%1.1f%%')
axes[1, 1].set_title('Distribution des catégories')
axes[1, 1].set_ylabel('')

plt.tight_layout()
plt.show()

In [ ]:
# Questions du dashboard :
print("--- Analyse du Dashboard ---")

# 1. Combien d'outliers identifiez-vous dans les âges ?
outliers_age = df_dashboard[(df_dashboard['age'] < 0) | (df_dashboard['age'] > 120)]
print(f"\n1. Outliers dans les âges : {len(outliers_age)}")
print(f"   Valeurs : {outliers_age['age'].values}")

# 2. Quel pourcentage de revenus est manquant ?
pct_revenu_missing = df_dashboard['revenu'].isnull().mean() * 100
print(f"\n2. Revenus manquants : {pct_revenu_missing:.1f}%")

# 3. Quel problème voyez-vous dans les catégories ?
pct_cat_missing = df_dashboard['categorie'].isnull().mean() * 100
print(f"\n3. Problème catégories : {pct_cat_missing:.1f}% de valeurs manquantes (None)")

---

## 4.8 Documentation des problèmes

### Créer une checklist de qualité

In [ ]:
def generer_rapport_qualite(df):
    """Génère un rapport de qualité complet."""
    rapport = []
    
    # 1. Complétude
    for col in df.columns:
        missing_pct = df[col].isnull().mean() * 100
        if missing_pct > 0:
            rapport.append({
                'Dimension': 'Complétude',
                'Colonne': col,
                'Problème': f'{missing_pct:.1f}% manquant',
                'Priorité': 'Haute' if missing_pct > 20 else 'Moyenne' if missing_pct > 5 else 'Basse'
            })
    
    # 2. Unicité
    doublons = df.duplicated().sum()
    if doublons > 0:
        rapport.append({
            'Dimension': 'Unicité',
            'Colonne': 'Toutes',
            'Problème': f'{doublons} doublons exacts',
            'Priorité': 'Haute'
        })
    
    # 3. Exactitude (numériques)
    for col in df.select_dtypes(include=[np.number]).columns:
        Q1, Q3 = df[col].quantile([0.25, 0.75])
        IQR = Q3 - Q1
        if IQR > 0:  # Éviter division par zéro
            outliers = df[(df[col] < Q1 - 1.5*IQR) | (df[col] > Q3 + 1.5*IQR)][col]
            if len(outliers) > 0:
                rapport.append({
                    'Dimension': 'Exactitude',
                    'Colonne': col,
                    'Problème': f'{len(outliers)} outliers détectés',
                    'Priorité': 'Moyenne'
                })
    
    return pd.DataFrame(rapport)

# Test de la fonction
print("Fonction de rapport créée.")

In [ ]:
# Générer le rapport sur notre DataFrame de dashboard
rapport = generer_rapport_qualite(df_dashboard)
print("Rapport de qualité :")
rapport

### Priorisation des problèmes

| Priorité | Critères | Action |
|----------|----------|--------|
| **🔴 Haute** | >20% missing, doublons, erreurs critiques | Corriger immédiatement |
| **🟡 Moyenne** | 5-20% missing, outliers à vérifier | Corriger si temps disponible |
| **🟢 Basse** | <5% missing, problèmes mineurs | Documenter pour plus tard |

### ✍️ Exercice 4.8 : Rapport de qualité complet (15 min)

In [ ]:
import pandas as pd
import numpy as np
import re

df_rapport = pd.DataFrame({
    'id': [1, 2, 3, 4, 5, 1, 7, 8, 9, 10],
    'nom': ['Alice', 'Bob', np.nan, 'David', 'Eve', 'Alice', 'Grace', 'Henry', np.nan, 'Julia'],
    'age': [25, 30, 35, 150, 28, 25, 42, -5, 38, 45],
    'email': ['a@t.com', 'b@t.com', 'c@t.com', 'd@t.com', 'invalid', 'a@t.com', 'g@t.com', 'h@t.com', 'i@t.com', 'j@t.com'],
    'salaire': [50000, 55000, np.nan, np.nan, 45000, 50000, 70000, 62000, np.nan, 80000]
})

print("DataFrame pour le rapport :")
df_rapport

In [ ]:
# Votre rapport complet
problemes = []

# 1. Complétude
print("=== 1. COMPLÉTUDE ===")
for col in df_rapport.columns:
    missing = df_rapport[col].isnull().sum()
    if missing > 0:
        pct = (missing / len(df_rapport)) * 100
        priorite = 'Haute' if pct > 20 else 'Moyenne'
        print(f"  {col}: {missing} manquant(s) ({pct:.0f}%) - Priorité {priorite}")
        problemes.append({'Type': 'Complétude', 'Colonne': col, 'Détail': f'{pct:.0f}% manquant', 'Priorité': priorite})

In [ ]:
# 2. Unicité
print("\n=== 2. UNICITÉ ===")
doublons_exact = df_rapport.duplicated().sum()
doublons_id = df_rapport.duplicated(subset=['id']).sum()
doublons_email = df_rapport.duplicated(subset=['email']).sum()

print(f"  Doublons exacts : {doublons_exact}")
print(f"  Doublons sur id : {doublons_id}")
print(f"  Doublons sur email : {doublons_email}")

if doublons_id > 0:
    problemes.append({'Type': 'Unicité', 'Colonne': 'id', 'Détail': f'{doublons_id} doublon(s)', 'Priorité': 'Haute'})

In [ ]:
# 3. Exactitude
print("\n=== 3. EXACTITUDE ===")
ages_invalides = df_rapport[(df_rapport['age'] < 0) | (df_rapport['age'] > 120)]
print(f"  Âges invalides (<0 ou >120) : {len(ages_invalides)}")
print(f"  Valeurs : {ages_invalides['age'].tolist()}")

if len(ages_invalides) > 0:
    problemes.append({'Type': 'Exactitude', 'Colonne': 'age', 'Détail': f'{len(ages_invalides)} invalide(s)', 'Priorité': 'Haute'})

In [ ]:
# 4. Validité (email)
print("\n=== 4. VALIDITÉ ===")
pattern_email = r'^[\w\.-]+@[\w\.-]+\.\w+$'
emails_invalides = df_rapport[~df_rapport['email'].str.match(pattern_email, na=False)]
print(f"  Emails invalides : {len(emails_invalides)}")
print(f"  Valeurs : {emails_invalides['email'].tolist()}")

if len(emails_invalides) > 0:
    problemes.append({'Type': 'Validité', 'Colonne': 'email', 'Détail': f'{len(emails_invalides)} invalide(s)', 'Priorité': 'Moyenne'})

In [ ]:
# Rapport final priorisé
print("\n" + "="*50)
print("RAPPORT DE QUALITÉ PRIORISÉ")
print("="*50)

rapport_final = pd.DataFrame(problemes)
rapport_final = rapport_final.sort_values('Priorité')
rapport_final

---

## 🧠 Réflexion métacognitive

### Auto-évaluation

| Compétence | 1 | 2 | 3 | 4 | 5 |
|------------|---|---|---|---|---|
| Je distingue EDA diagnostique et analytique | ○ | ○ | ○ | ○ | ○ |
| Je connais les 6 dimensions de qualité | ○ | ○ | ○ | ○ | ○ |
| Je sais détecter les valeurs manquantes | ○ | ○ | ○ | ○ | ○ |
| Je sais identifier les doublons | ○ | ○ | ○ | ○ | ○ |
| Je sais détecter les outliers (IQR, Z-score) | ○ | ○ | ○ | ○ | ○ |
| Je peux créer un rapport de qualité priorisé | ○ | ○ | ○ | ○ | ○ |

### Questions de réflexion

1. **Quelle dimension de qualité** vous semble la plus difficile à évaluer ? Pourquoi ?

2. **Comment expliqueriez-vous** l'importance de l'EDA diagnostique à un manager non-technique ?

3. **Dans votre futur métier**, quel type de problème de qualité pensez-vous rencontrer le plus souvent ?

---

## 📚 Résumé du chapitre

### Points clés à retenir

1. **EDA Diagnostique ≠ Analytique** :
   - Diagnostique = trouver les problèmes (avant nettoyage)
   - Analytique = comprendre les patterns (après nettoyage)

2. **6 dimensions de qualité** :
   - Complétude, Unicité, Cohérence
   - Exactitude, Validité, Fraîcheur

3. **Outils de détection** :
   - Missing : `isnull()`, `missingno`
   - Doublons : `duplicated()`
   - Outliers : IQR (robuste), Z-score (distributions normales)

4. **Documentation** :
   - Toujours créer une checklist de qualité
   - Prioriser : Haute > Moyenne > Basse

---

## 🔗 Sources et références

- [IBM - Data Quality Dimensions](https://www.ibm.com/think/topics/data-quality-dimensions)
- [Atlan - Data Quality Dimensions 2025](https://atlan.com/data-quality-dimensions/)
- [Monte Carlo - 6 Data Quality Dimensions](https://www.montecarlodata.com/blog-6-data-quality-dimensions-examples/)
- [dbt Labs - Data Quality Dimensions](https://www.getdbt.com/blog/data-quality-dimensions)

---

## ➡️ Prochain chapitre

**Chapitre 5 : Nettoyage des données** — Vous apprendrez à corriger les problèmes identifiés : traiter les valeurs manquantes, supprimer les doublons, et gérer les outliers.

---

*Module 2 — Pipeline Data | Chapitre 4 sur 11*